In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import json
from io import StringIO
import statsmodels.api as sm #ARIMAX - https://people.duke.edu/~rnau/411arim.htm?utm_source=copilot.com # I plan to use p and q max of 5 based on the defaults in the r package auto.arima (https://pkg.robjhyndman.com/forecast/reference/auto.arima.html?utm_source=copilot.com)
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

In [2]:
# Preset alpha value for the Bernferroni correction
alpha = 0.05

In [3]:
# Parent directory for data loading
parent_dir = Path.cwd().parent.parent

In [4]:
# Import datasets and drop any nulls for clean processing
final_gig_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'final', 'gig_monthly_data.csv')).dropna().reset_index(drop=True)
final_sp500_m_df = pd.read_csv(os.path.join(parent_dir, 'data', 'final', 'sp500_monthly_data.csv')).dropna().reset_index(drop=True)

In [ ]:
# See correlation_analysis_for_regression.ipynb for how these columns were selected
# Partially based on correlation and partially based on an understanding of the different meanings behind economic indicators
columns_to_keep = ['cpi_data_monthly', # SA
'rec_smooth_prob_data_monthly', # NSA -- but I don't think recessions are seasonal
'unemployment_data_monthly', # SA
'interest_rate_mtg_data_weekly', # NSA
'consumer_sentiment_data_monthly', # NSA
'target'#, # Normalized by dividing closing price with cpi
]

In [ ]:
# Reduce datasets to only use columns to keep and set date as index
final_gig_m_df.set_index('date', inplace=True)
final_gig_m_df = final_gig_m_df[columns_to_keep]
final_sp500_m_df.set_index('date', inplace=True)
final_sp500_m_df = final_sp500_m_df[columns_to_keep]

In [ ]:
# In order to run the ARIMAX regression, I need to check for stationarity among all variables, including the endogenous variable (target)
# Interpreting the Results - Taken from (https://www.geeksforgeeks.org/machine-learning/augmented-dickey-fuller-adf/)
# - ADF Statistic: If it's significantly lower than the critical values, reject the null.
# - p-value: If p < 0.05, the series is likely stationary.
# - Critical Values: Used for comparing with the ADF statistic.
# To fix it, we will be running the diff function on the column per https://www.geeksforgeeks.org/machine-learning/time-series-data-transformation-using-python/
for col in columns_to_keep:

    adf = sm.tsa.stattools.adfuller(final_sp500_m_df[col])
    adf_dict = {'column':col, 'adf-score': adf[0], 'p-value': adf[1], 'critical-values': adf[4]}
    print(adf_dict)

    if adf_dict['p-value'] > alpha \
            and adf_dict['adf-score'] > adf_dict['critical-values']['1%'] \
            and adf_dict['adf-score'] > adf_dict['critical-values']['5%'] \
            and adf_dict['adf-score'] > adf_dict['critical-values']['10%']:
        
        final_sp500_m_df[col] = final_sp500_m_df[col].diff()
        final_sp500_m_df.dropna(inplace=True)
        final_gig_m_df[col] = final_gig_m_df[col].diff()
        final_gig_m_df.dropna(inplace=True)

        adf = sm.tsa.stattools.adfuller(final_sp500_m_df[col])
        adf_dict = {'column':col, 'adf-score': adf[0], 'p-value': adf[1], 'critical-values': adf[4]}
        print(adf_dict)

        if adf_dict['p-value'] > alpha \
                and adf_dict['adf-score'] > adf_dict['critical-values']['1%'] \
                and adf_dict['adf-score'] > adf_dict['critical-values']['5%'] \
                and adf_dict['adf-score'] > adf_dict['critical-values']['10%']:
            
            print(f'{col} is broke!')

{'column': 'cpi_data_monthly', 'adf-score': np.float64(3.6385924285269846), 'p-value': 1.0, 'critical-values': {'1%': np.float64(-3.4304329171888397), '5%': np.float64(-2.8615766479902), '10%': np.float64(-2.566789506461638)}}
{'column': 'cpi_data_monthly', 'adf-score': np.float64(-281.0463385740601), 'p-value': 0.0, 'critical-values': {'1%': np.float64(-3.4304329182402284), '5%': np.float64(-2.8615766484548897), '10%': np.float64(-2.5667895067089774)}}
{'column': 'rec_smooth_prob_data_monthly', 'adf-score': np.float64(-6.442359071026206), 'p-value': np.float64(1.596490138072952e-08), 'critical-values': {'1%': np.float64(-3.4304329182402284), '5%': np.float64(-2.8615766484548897), '10%': np.float64(-2.5667895067089774)}}
{'column': 'unemployment_data_monthly', 'adf-score': np.float64(-2.7153602541006316), 'p-value': np.float64(0.07141778120614303), 'critical-values': {'1%': np.float64(-3.4304329182402284), '5%': np.float64(-2.8615766484548897), '10%': np.float64(-2.5667895067089774)}}


In [19]:
# Get all possible combinations of exogenous variables for the regression analysis
# Since both datasets have the same columns, we can just use the columns from one of them to get the combinations
exog_cols = []
for c in final_gig_m_df.columns:
    temp_cols = []
    if c != 'target':
        temp_cols.append(c)
        for c2 in final_gig_m_df.columns:
            if c2 != 'target' and c2 != c:
                temp_cols.append(c2)
                temp_col_set = frozenset(sorted(temp_cols))
                exog_cols.append(temp_col_set)
exog_cols = frozenset(exog_cols)
exog_cols = [sorted(i) for i in exog_cols]

In [14]:
# Modify the alpha to use the bonferroni correction since we are testing many combinations
bonferroni_alpha = alpha / len(exog_cols)

In [ ]:
# Run ARIMAX regression analysis for each combination of exogenous variables and iterate over the different p and q values (0-5) to find the best model based on AIC/BIC for each column combination
for exog_col in exog_cols:

    for p in range(0,6):
        for q in range(0,6):

            # Run regression analysis for gig dataset
            model_gig = sm.tsa.statespace.SARIMAX(final_gig_m_df['target'], exog=final_gig_m_df[exog_col], order=(p,0,q))
            results_gig = model_gig.fit()
            
            # Run regression analysis for sp500 dataset
            model_sp500 = sm.tsa.statespace.SARIMAX(final_sp500_m_df['target'], exog=final_sp500_m_df[exog_col], order=(p,0,q))
            results_sp500 = model_sp500.fit()